# Forklift Feature-Based Anomaly Detection PoC

`data/movie_preprocess/normal` にある正常動画だけを使い、音声特徴・オプティカルフロー特徴・グローバルモーション特徴から正常分布を学習する PoC ノートブックです。

今回の初期実装ではセグメンテーションや RGB 画像そのものの深層学習は扱わず、CPU で動かしやすい OpenCV + librosa + scikit-learn の特徴量ベース構成にします。


大きな流れは、1. 正常データの探索、2. 音声特徴抽出、3. optical flow から広域振動スコア作成、4. audio/motion の IsolationForest 学習、5. スコア合成と artifact 保存です。推論ノートが同じ特徴量定義を再利用できるよう、settings と列順も保存します。


## 0. パッケージ確認

必要なパッケージが未導入の場合は、下のコメントを外してインストールしてください。音声抽出済みの `*_audio.wav` を使うため、通常は `ffmpeg` がなくても学習まで進められます。

In [ ]:
# ------------------------------------------------------------
# セル概要: パッケージ導入
# ------------------------------------------------------------
# - 学習PoCで使う OpenCV / librosa / scikit-learn などをインストールします。
# - 環境が固定されている場合は再実行不要ですが、Notebook単体で再現しやすいよう残しています。
# ------------------------------------------------------------

%pip install -q opencv-python numpy pandas scipy librosa scikit-learn tqdm joblib

## 1. ライブラリ読み込み

In [ ]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - 画像処理、音声処理、表処理、IsolationForest 学習に必要なライブラリを読み込みます。
# - librosa / tqdm がない環境でも処理が落ちにくいようフォールバックを用意しています。
# ------------------------------------------------------------

# ============================================================
# ライブラリ読み込み
# ------------------------------------------------------------
# このノートブックで使う標準ライブラリ、画像処理、音声処理、
# 機械学習、描画系のライブラリをまとめて読み込みます。
# tqdm / librosa は環境によって入っていない可能性があるため、
# import に失敗しても最低限処理が続くようフォールバックしています。
# ============================================================
from __future__ import annotations

import math
import warnings
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample_poly
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    import librosa
except Exception as exc:
    librosa = None
    warnings.warn(f"librosa could not be imported. Audio features will use a scipy fallback. reason={exc}")

pd.set_option("display.max_columns", 120)


## 2. 設定値

最初は `MAX_TRAIN_VIDEOS` を小さめにして特徴量の妥当性と処理時間を確認し、問題なければ `None` にして正常データ全体で学習します。

In [ ]:
# ------------------------------------------------------------
# セル概要: 学習設定
# ------------------------------------------------------------
# - 学習データ、サンプリング窓、flow グリッド、スコア合成、保存先をまとめます。
# - 推論ノートは保存された settings を読み戻すため、ここが学習・推論の仕様の中心です。
# ------------------------------------------------------------

# ============================================================
# 共通設定
# ------------------------------------------------------------
# 学習ノート全体で使うパス、サンプリング間隔、モデル設定、
# 広域振動スコアの計算パラメータをここに集約しています。
# 推論ノートは、ここで保存した settings を読み取って同じ条件で
# 特徴量を作るため、学習と推論で値を揃えることが重要です。
# ============================================================
# 関数メモ: find_project_root
# - 現在の作業ディレクトリから親方向へたどり、リポジトリの基準ディレクトリを見つけます。
# - Notebook を `notebooks/` から実行しても repo root から実行しても、入出力パスがずれないようにします。

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "movie_preprocess").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}.")


# ノートブックをどこから実行しても repo root を見つけ、入出力パスを絶対化します。
PROJECT_ROOT = find_project_root()
TRAIN_DATA_DIR = PROJECT_ROOT / "data" / "movie_preprocess" / "normal"
MANIFEST_PATH = PROJECT_ROOT / "data" / "movie_preprocess" / "movie_preprocess_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_feature_poc"
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / "data" / "train_sample_list.csv"

# 動画と音声を 0.2 秒窓でそろえるための基本設定です。
FPS_SAMPLE = 10
WINDOW_SEC = 0.2
AUDIO_SR = 16000
N_MELS = 16

USE_FRONT = True
USE_REAR = True
FRAME_RESIZE_WIDTH = 480
FLOW_ANALYSIS_SCALE = 0.75
FLOW_GRID = (3, 3)
RANDOM_STATE = 42
MAX_TRAIN_VIDEOS = 50

# 学習対象は normal のみです。manual は TRAIN_SPECS の sample_id を使い、
# random は normal の usable データから件数と indoor/outdoor 比率に従って選びます。
TRAIN_SELECTION_MODE = "random"  # "manual" or "random"
TRAIN_SPECS = [
    # {"sample_id": "1001"},
    # {"sample_id": "1024"},
]
# RANDOM_TRAIN_COUNT < 0 の場合は、usable な normal データを全件使います。
RANDOM_TRAIN_COUNT = MAX_TRAIN_VIDEOS
RANDOM_TRAIN_ENVIRONMENT_RATIOS = {"indoor": 0.8, "outdoor": 0.2}

CONTAMINATION = "auto"

# 広域振動スコアは 1.0 秒窓を 0.5 秒刻みでずらして計算します。
# つまり、モデルへ入る motion 特徴は衝撃前後の短い時間変化を少し広めに見る設計です。
FLOW_SCORE_WINDOW_SEC = 1.0
FLOW_SCORE_HOP_SEC = 0.5
FLOW_SCORE_ALPHA_MIN = 0.04
FLOW_SCORE_ALPHA_MAX = 0.42
FLOW_SCORE_MIN_VISIBLE = 1e-6
FLOW_SCORE_HIGH_RATIO_FRACTION = 0.5
# flow の大きさが小さすぎる点は向きが不安定なので、向き変化の計算から除外します。
FLOW_DIRECTION_MIN_MAG = 0.025
DIRECTION_JITTER_HIGH_THRESHOLD = 1.5
DIRECTION_JITTER_LOW_THRESHOLD = 0.8
# 振動候補の抽出閾値は固定値ではなく、各グリッド内スコアのパーセンタイルで決めます。
DIRECTION_JITTER_THRESHOLD_MODE = "percentile"
DIRECTION_JITTER_HIGH_PERCENTILE = 85.0
DIRECTION_JITTER_LOW_PERCENTILE = 60.0
DIRECTION_JITTER_LOW_EXPANSION_STEPS = 2
DIRECTION_JITTER_SCORE_LOWER_PERCENTILE = 0.0
DIRECTION_JITTER_SCORE_UPPER_PERCENTILE = 95.0
# 同じ時刻の全グリッドのうち、閾値通過後の上位 K 個を平均して広域振動スコアにします。
BROAD_VIBRATION_TOP_K = 5
BROAD_VIBRATION_COLUMNS = ["front_broad_vibration_score", "rear_broad_vibration_score"]

# モデルに入れる特徴量グループです。
# audio モデルは音声のみ、motion モデルは front/rear の広域振動スコアのみを使います。
FEATURE_GROUPS = {
    "audio_basic": True,
    "audio_mel": True,
    "broad_vibration": True,
}
FEATURE_EXCLUDE_COLUMNS: list[str] = []
FEATURE_GROUP_WEIGHTS = {
    "audio_basic": 1.0,
    "audio_mel": 1.0,
    "broad_vibration": 1.0,
}

# IsolationForest は音声用と動き用を別々に学習します。
# それぞれの異常度を後段で同期スコアと合成します。
SCORE_MODEL_CONFIGS = {
    "audio": {
        "enabled": True,
        "score_column": "audio_anomaly_score",
        "raw_score_column": "audio_anomaly_score_raw",
        "feature_groups": {"audio_basic": True, "audio_mel": True, "broad_vibration": False},
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
    },
    "motion": {
        "enabled": True,
        "score_column": "motion_anomaly_score",
        "raw_score_column": "motion_anomaly_score_raw",
        "feature_groups": {"audio_basic": False, "audio_mel": False, "broad_vibration": True},
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
    },
}

SCORE_CALIBRATION_QUANTILES = (0.01, 0.99)
# 音声ピークと動きピークが近い時刻に出ているかを見る設定です。
# WINDOW_SEC=0.2 のため max_lag_windows=2 は前後 0.4 秒を許容します。
SYNC_SCORE_CONFIG = {
    "audio_score_column": "audio_anomaly_score",
    "motion_score_column": "motion_anomaly_score",
    "max_lag_windows": 2,
}
# 最終異常スコア = 音声異常度 + 動き異常度 + 同期スコア の重み付き平均です。
FINAL_SCORE_WEIGHTS = {
    "audio_anomaly_score": 0.45,
    "motion_anomaly_score": 0.35,
    "sync_score": 0.20,
}

PLOT_SCORE_COLUMNS = ["anomaly_score", "audio_anomaly_score", "motion_anomaly_score", "sync_score"]
PLOT_FEATURE_COLUMNS = [
    "audio_rms",
    "audio_high_freq_energy",
    "front_broad_vibration_score",
    "rear_broad_vibration_score",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TRAIN_DATA_DIR: {TRAIN_DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print("broad vibration columns:", BROAD_VIBRATION_COLUMNS)
print("broad vibration top_k:", BROAD_VIBRATION_TOP_K)


## 3. 正常データの検出

前処理済みデータは `sample_id_front.mp4` / `sample_id_rear.mp4` / `sample_id_audio.wav` の3ファイルを1サンプルとして扱います。

In [ ]:
# ------------------------------------------------------------
# セル概要: 正常データ一覧の作成
# ------------------------------------------------------------
# - 前処理済み normal データから front/rear/audio がそろったサンプルを探します。
# - manifest があれば環境や前処理状態も取り込み、usable フラグで学習可能性を判定します。
# ------------------------------------------------------------

# ============================================================
# 学習対象データの収集
# ------------------------------------------------------------
# data/movie_preprocess/normal 以下から、front/rear/audio がそろった
# 正常動画だけを探します。anomaly は学習対象にしません。
# manifest がある場合は environment などのメタ情報も付与します。
# ============================================================
# 関数メモ: discover_movie_preprocess_samples
# - 前処理済み normal データから、front/rear/audio がそろう学習候補を一覧化します。
# - manifest があれば処理状態や環境情報を結合し、usable フラグを作ります。

def discover_movie_preprocess_samples(data_dir: Path, manifest_path: Path | None = None) -> pd.DataFrame:
    rows = []
    for env_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        by_id: dict[str, dict] = {}
        for path in sorted(env_dir.glob("*_*.*")):
            stem = path.stem
            if "_" not in stem:
                continue
            sample_id, kind = stem.rsplit("_", 1)
            if kind not in {"front", "rear", "audio"}:
                continue
            row = by_id.setdefault(sample_id, {"sample_id": sample_id, "category": "normal", "environment": env_dir.name})
            row[f"{kind}_path"] = path
        rows.extend(by_id.values())

    samples = pd.DataFrame(rows)
    if samples.empty:
        return samples

    for col in ["front_path", "rear_path", "audio_path"]:
        if col not in samples.columns:
            samples[col] = pd.NA

    if manifest_path and manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        manifest = manifest[manifest["category"].astype(str).eq("normal")].copy()
        if "front_output_path" in manifest.columns:
            manifest["sample_id"] = manifest["front_output_path"].astype(str).map(lambda value: Path(value).name.removesuffix("_front.mp4"))
        else:
            manifest["sample_id"] = manifest["input_video_path"].astype(str).str.extract(r"(\d+)")
        keep_cols = [
            "sample_id", "category", "environment", "input_duration_sec", "written_frame_pairs",
            "trim_start_sec", "trim_duration_sec", "status", "audio_status", "output_width", "output_height",
        ]
        manifest = manifest[[c for c in keep_cols if c in manifest.columns]].drop_duplicates(["sample_id", "environment"])
        samples = samples.merge(manifest, on=["sample_id", "environment"], how="left", suffixes=("", "_manifest"))
        if "category_manifest" in samples.columns:
            samples["category"] = samples["category"].fillna(samples["category_manifest"]).fillna("normal")
            samples = samples.drop(columns=["category_manifest"])

    samples["has_front"] = samples["front_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["has_rear"] = samples["rear_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["has_audio"] = samples["audio_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["usable"] = samples[["has_front", "has_rear", "has_audio"]].all(axis=1)
    return samples.sort_values(["environment", "sample_id"]).reset_index(drop=True)

samples_df = discover_movie_preprocess_samples(TRAIN_DATA_DIR, MANIFEST_PATH)
print(samples_df.shape)
display(samples_df.head(10))
print(samples_df[["category", "environment", "usable"]].value_counts(dropna=False))

## 4. 動画読み込みユーティリティ


In [ ]:
# ------------------------------------------------------------
# セル概要: 動画読み込み
# ------------------------------------------------------------
# - 動画を指定FPSで間引き、必要なら横幅を縮小して optical flow の計算負荷を下げます。
# - フレームごとに時刻を持たせ、音声特徴と同じ time_bin に結合できるようにします。
# ------------------------------------------------------------

# ============================================================
# 動画・音声の読み込み
# ------------------------------------------------------------
# 動画は一定 FPS で間引き、必要なら横幅を縮小して処理量を抑えます。
# 音声は 0.2 秒窓へ分割し、RMS や log-mel などの特徴を作ります。
# ============================================================
# アスペクト比を保ったまま横幅だけをそろえます。flow の閾値を動画間で比較しやすくするためです。
# 関数メモ: resize_keep_aspect
# - アスペクト比を保ったまま横幅だけを縮小します。
# - 動画ごとの見た目を壊さず、optical flow の処理量とスケールをそろえます。

def resize_keep_aspect(frame: np.ndarray, width: int | None) -> np.ndarray:
    if width is None or frame.shape[1] <= width:
        return frame
    scale = width / frame.shape[1]
    height = max(1, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


# 指定 FPS になるようにフレームを間引いて読み込みます。各行に時刻と画像を持たせます。
# 関数メモ: extract_video_frames
# - 動画を指定FPS相当で間引いて読み込み、時刻付きフレームリストを作ります。
# - 音声や広域振動スコアと同じ時間軸へ後で結合するため、各フレームに `time` を持たせます。

def extract_video_frames(
    video_path: str | Path,
    fps_sample: float = FPS_SAMPLE,
    resize_width: int | None = FRAME_RESIZE_WIDTH,
    max_duration_sec: float | None = None,
) -> list[dict]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        warnings.warn(f"Could not open video: {video_path}")
        return []
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = fps_sample
        # 元動画 FPS から、何フレームおきに読むかを計算します。
    frame_interval = max(1, int(round(src_fps / fps_sample)))

    frames = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        time_sec = frame_idx / src_fps
        if max_duration_sec is not None and time_sec > max_duration_sec:
            break
        if frame_idx % frame_interval == 0:
            frames.append({"time": time_sec, "frame": resize_keep_aspect(frame, resize_width)})
        frame_idx += 1
    cap.release()
    return frames


## 5. 音声特徴抽出

`*_audio.wav` から 0.2 秒窓の音声特徴を作ります。

In [ ]:
# ------------------------------------------------------------
# セル概要: 音声特徴量抽出
# ------------------------------------------------------------
# - wav をモノラル化し、0.2秒窓ごとにRMS・ピーク・周波数重心・log-melを作ります。
# - audio モデルは主にこの特徴群から衝撃音や異音らしさを学習します。
# ------------------------------------------------------------

# ============================================================
# 音声特徴量
# ------------------------------------------------------------
# 音声をモノラル化して AUDIO_SR にそろえ、WINDOW_SEC ごとに
# 基本統計と log-mel 特徴を作ります。audio モデルはこの列だけで学習します。
# ============================================================
# librosa が使える場合は librosa、なければ scipy で wav を読みます。
# 関数メモ: load_audio_mono
# - wav をモノラル float32 に変換し、必要なら指定サンプリング周波数へリサンプリングします。
# - 学習時と推論時で音声特徴の窓幅・周波数軸を一致させるための入口です。

def load_audio_mono(wav_path: str | Path, sr: int = AUDIO_SR) -> tuple[np.ndarray, int]:
    wav_path = Path(wav_path)
    if not wav_path.exists():
        return np.array([], dtype=np.float32), sr
    if librosa is not None:
        y, used_sr = librosa.load(wav_path, sr=sr, mono=True)
        return y.astype(np.float32), used_sr

    used_sr, y = wavfile.read(wav_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    max_abs = np.max(np.abs(y)) if y.size else 1.0
    if max_abs > 0:
        y = y / max_abs
    if used_sr != sr and y.size:
        gcd = math.gcd(used_sr, sr)
        y = resample_poly(y, sr // gcd, used_sr // gcd).astype(np.float32)
        used_sr = sr
    return y, used_sr


# 関数メモ: extract_audio_features
# - 音声を固定時間窓に分け、RMS・ピーク・ZCR・周波数統計・log-mel を作ります。
# - 音声由来の異常度を学習・推論するための基本特徴表を返します。

def extract_audio_features(
    wav_path: str | Path,
    window_sec: float = WINDOW_SEC,
    sr: int = AUDIO_SR,
    n_mels: int = N_MELS,
    video_id: str | None = None,
) -> pd.DataFrame:
    y, used_sr = load_audio_mono(wav_path, sr=sr)
    win = max(1, int(round(window_sec * used_sr)))
    if y.size == 0:
        return pd.DataFrame(columns=["video_id", "time_bin", "time", "audio_missing"])

    rows = []
        # 音声波形を 0.2 秒窓に分け、各窓がモデルの1サンプルになります。
    for i in range(int(math.ceil(len(y) / win))):
        segment = y[i * win : min(len(y), (i + 1) * win)]
        if segment.size == 0:
            continue
                # FFT から周波数重心・帯域幅・高周波エネルギーを計算します。
        spectrum = np.abs(np.fft.rfft(segment * np.hanning(segment.size)))
        freqs = np.fft.rfftfreq(segment.size, d=1.0 / used_sr)
        power = spectrum ** 2
        power_sum = float(power.sum()) + 1e-12
        centroid = float((freqs * power).sum() / power_sum)
        rows.append({
            "video_id": video_id,
            "time_bin": i,
            "time": i * window_sec,
            "audio_rms": float(np.sqrt(np.mean(segment ** 2))),
            "audio_energy": float(np.mean(segment ** 2)),
            "audio_peak": float(np.max(np.abs(segment))),
            "audio_ptp": float(np.ptp(segment)),
            "audio_zcr": float(np.mean(np.abs(np.diff(np.signbit(segment))).astype(float))) if segment.size > 1 else 0.0,
            "audio_centroid": centroid,
            "audio_bandwidth": float(np.sqrt((((freqs - centroid) ** 2) * power).sum() / power_sum)),
            "audio_high_freq_energy": float(power[freqs >= 3000].sum() / power_sum),
            "audio_missing": 0,
        })
    df = pd.DataFrame(rows)

        # log-mel は音色や衝撃音の違いを表現するための追加特徴です。
    if librosa is not None and len(df):
        mel = librosa.feature.melspectrogram(
            y=y, sr=used_sr, n_mels=n_mels, n_fft=min(2048, max(256, win)), hop_length=win, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).T
        for m in range(n_mels):
            values = np.full(len(df), np.nan, dtype=float)
            values[: min(len(values), log_mel.shape[0])] = log_mel[: min(len(values), log_mel.shape[0]), m]
            df[f"audio_mel_{m:02d}"] = values
    else:
        for m in range(n_mels):
            df[f"audio_mel_{m:02d}"] = 0.0
    return df


## 6. オプティカルフロー特徴抽出

Farneback 法でフローを計算し、画面全体と 3x3 グリッドの統計量に落とします。

In [ ]:
# ------------------------------------------------------------
# セル概要: オプティカルフロー特徴
# ------------------------------------------------------------
# - front/rear 動画から Farneback optical flow を計算し、全体統計と3x3グリッド統計を作ります。
# - 後段の広域振動スコアではグリッドごとの x/y 平均を使って向きの揺れを評価します。
# ------------------------------------------------------------

# ============================================================
# optical flow 特徴量
# ------------------------------------------------------------
# Farneback optical flow で、各フレーム間の画素移動量を推定します。
# 後段の広域振動スコアでは、3x3 グリッドごとの flow x/y 平均を使い、
# 向きが小刻みに変化しているかを評価します。
# ============================================================
# 現時点では画面全体を有効領域として扱います。
# 将来フォークリフト車体や固定UIなどを除外したくなった場合は、
# compute_optical_flow_features 内の valid_mask 作成箇所を差し替えます。
# 関数メモ: masked_values
# - 有効マスク内の値だけを取り出し、有限値に絞ります。
# - マスクが空の場合は必要に応じて全画素へフォールバックし、統計値が欠落しにくくします。

def masked_values(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> np.ndarray:
    selected = values[valid_mask]
    if selected.size == 0 and fallback_to_all:
        selected = values.reshape(-1)
    return selected[np.isfinite(selected)]


# 関数メモ: masked_mean
# - マスク済み値の平均を返します。
# - 対象領域が空でも 0.0 を返し、特徴量テーブルの列構造を保ちます。

def masked_mean(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.mean(selected)) if selected.size else 0.0


# 関数メモ: masked_std
# - マスク済み値の標準偏差を返します。
# - flow のばらつきを特徴化しつつ、空領域では安全に 0.0 にします。

def masked_std(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.std(selected)) if selected.size else 0.0




# Farneback の計算負荷を下げるため、解析用グレースケール画像を縮小します。
# 関数メモ: resize_gray_for_flow
# - flow 計算用のグレースケール画像を縮小し、元サイズ換算に戻すための倍率も返します。
# - Farneback の計算負荷を抑えつつ、出力 flow を元のリサイズ済み動画スケールで解釈します。

def resize_gray_for_flow(gray: np.ndarray, scale: float = FLOW_ANALYSIS_SCALE) -> tuple[np.ndarray, float, float]:
    scale = float(scale)
    if scale <= 0.0 or scale >= 1.0:
        return gray, 1.0, 1.0
    h, w = gray.shape[:2]
    new_w = max(8, int(round(w * scale)))
    new_h = max(8, int(round(h * scale)))
    if new_w == w and new_h == h:
        return gray, 1.0, 1.0
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized, new_w / max(w, 1), new_h / max(h, 1)


# flow 計算に失敗したフレームでも列構造を崩さないため、0 埋め行を返します。
# 関数メモ: _empty_flow_row
# - flow 計算に失敗した時刻でも、期待される列を0埋めで返します。
# - 一部フレームの失敗で DataFrame の列が欠け、後続の結合やモデル入力が壊れるのを防ぎます。

def _empty_flow_row(prefix: str, time_sec: float, time_bin: int, video_id: str | None) -> dict:
    row = {
        "video_id": video_id,
        "time_bin": time_bin,
        "time": time_sec,
        f"{prefix}_flow_mag_mean": 0.0,
        f"{prefix}_flow_mag_std": 0.0,
        f"{prefix}_flow_mag_max": 0.0,
        f"{prefix}_flow_angle_mean": 0.0,
        f"{prefix}_flow_angle_std": 0.0,
        f"{prefix}_flow_x_mean": 0.0,
        f"{prefix}_flow_x_std": 0.0,
        f"{prefix}_flow_y_mean": 0.0,
        f"{prefix}_flow_y_std": 0.0,
        f"{prefix}_flow_failed": 1.0,
    }
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            for name in ["mag_mean", "mag_std", "x_mean", "y_mean", "valid_ratio"]:
                row[f"{prefix}_flow_cell_{gy}_{gx}_{name}"] = 0.0
    return row


# 前フレームと現フレームの差分から、全体平均と 3x3 グリッド別の flow を作ります。
# 関数メモ: compute_optical_flow_features
# - 連続フレーム間の dense optical flow を計算し、全体統計とグリッド統計へ集約します。
# - front/rear それぞれの motion 特徴と、後段の広域振動スコアの材料を作ります。

def compute_optical_flow_features(
    frames: list[dict],
    prefix: str,
    window_sec: float = WINDOW_SEC,
    grid: tuple[int, int] = FLOW_GRID,
    video_id: str | None = None,
) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()

    rows = []
    prev_gray, prev_scale_x, prev_scale_y = resize_gray_for_flow(cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY))
    for item in frames[1:]:
        time_sec = float(item["time"])
        time_bin = int(round(time_sec / window_sec))
        try:
            gray, scale_x, scale_y = resize_gray_for_flow(cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY))
            # Farneback は各画素の移動ベクトルを dense に推定します。
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0,
            )
            # 縮小画像で計算した移動量を、元のリサイズ済み画像スケールへ戻します。
            fx = flow[..., 0] / max(scale_x, 1e-6)
            fy = flow[..., 1] / max(scale_y, 1e-6)
            mag, angle = cv2.cartToPolar(fx, fy, angleInDegrees=False)
            # 今は全画素を有効として扱います。valid_mask を1か所に閉じ込めておくと、
            # 後で「車体の映り込みを除外する」などの変更を入れやすくなります。
            valid_mask = np.ones(mag.shape, dtype=bool)
            mag_selected = masked_values(mag, valid_mask)
            row = {
                "video_id": video_id,
                "time_bin": time_bin,
                "time": time_sec,
                f"{prefix}_flow_mag_mean": masked_mean(mag, valid_mask),
                f"{prefix}_flow_mag_std": masked_std(mag, valid_mask),
                f"{prefix}_flow_mag_max": float(np.max(mag_selected)) if mag_selected.size else 0.0,
                f"{prefix}_flow_angle_mean": masked_mean(angle, valid_mask),
                f"{prefix}_flow_angle_std": masked_std(angle, valid_mask),
                f"{prefix}_flow_x_mean": masked_mean(fx, valid_mask),
                f"{prefix}_flow_x_std": masked_std(fx, valid_mask),
                f"{prefix}_flow_y_mean": masked_mean(fy, valid_mask),
                f"{prefix}_flow_y_std": masked_std(fy, valid_mask),
                f"{prefix}_flow_failed": 0.0,
            }
            h, w = mag.shape
            # 画面を 3x3 に分け、どの領域で振動が強いかを後から見られるようにします。
            for gy in range(grid[0]):
                y0, y1 = int(h * gy / grid[0]), int(h * (gy + 1) / grid[0])
                for gx in range(grid[1]):
                    x0, x1 = int(w * gx / grid[1]), int(w * (gx + 1) / grid[1])
                    cell_mask = valid_mask[y0:y1, x0:x1]
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_mean"] = masked_mean(mag[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_std"] = masked_std(mag[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_x_mean"] = masked_mean(fx[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_y_mean"] = masked_mean(fy[y0:y1, x0:x1], cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"] = float(np.mean(cell_mask)) if cell_mask.size else 0.0
        except Exception as exc:
            warnings.warn(f"{prefix} optical flow failed at {time_sec:.2f}s: {exc}")
            gray, scale_x, scale_y = resize_gray_for_flow(cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY))
            row = _empty_flow_row(prefix, time_sec, time_bin, video_id)
        rows.append(row)
        prev_gray, prev_scale_x, prev_scale_y = gray, scale_x, scale_y

    return pd.DataFrame(rows)


## 9. 特徴量の時間結合

In [ ]:
# ------------------------------------------------------------
# セル概要: 時間結合と広域振動
# ------------------------------------------------------------
# - 音声・flow・広域振動スコアを video_id + time_bin で結合します。
# - motion モデルに入れる `front_broad_vibration_score` / `rear_broad_vibration_score` もここで作ります。
# ------------------------------------------------------------

# ============================================================
# 広域振動スコアと特徴量結合
# ------------------------------------------------------------
# audio / flow など別々に作った特徴を時間窓で結合し、
# raw flow から「同時刻の上位グリッド平均」である広域振動スコアを作ります。
# motion モデルにはこの front/rear の広域振動スコアだけを入れます。
# ============================================================
# 各特徴ソースを video_id + time_bin ごとに1行へ集約します。
# 関数メモ: _aggregate_feature_df
# - 任意の特徴表を video_id + time_bin 単位へ集約します。
# - 同じ窓に複数行がある場合は数値列を平均し、文字列列は代表値を残します。

def _aggregate_feature_df(df: pd.DataFrame, window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    if "video_id" not in df.columns:
        df["video_id"] = "unknown"
    if "time_bin" not in df.columns:
        if "time" not in df.columns:
            raise ValueError("feature df must have either time_bin or time")
        df["time_bin"] = np.round(df["time"] / window_sec).astype(int)

    key_cols = ["video_id", "time_bin"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    other_cols = [c for c in df.columns if c not in set(key_cols + numeric_cols + ["time"])]
    out = df.groupby(key_cols, as_index=False)[numeric_cols].mean() if numeric_cols else df[key_cols].drop_duplicates()
    for col in other_cols:
        values = df.groupby(key_cols)[col].agg(lambda x: x.dropna().mode().iat[0] if len(x.dropna().mode()) else "unknown").reset_index()
        out = out.merge(values, on=key_cols, how="left")
    out["time"] = out["time_bin"] * window_sec
    return out


# 音声・flow・広域振動スコアを time_bin で outer join し、欠損は前後の値で補います。
# この関数の出力が「モデルが見る時間軸」です。WINDOW_SEC=0.2 なので、
# audio も motion も最終的には 0.2 秒ごとに1行の表へ変換されます。
# 関数メモ: align_features_by_time
# - 複数の特徴表を time_bin で outer join し、欠損値を動画内で前後補完します。
# - 音声0.2秒窓と広域振動0.5秒hopのような異なる時間解像度を、モデル入力の1表へそろえます。

def align_features_by_time(feature_dfs: list[pd.DataFrame], window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    valid = [_aggregate_feature_df(df, window_sec=window_sec) for df in feature_dfs if df is not None and len(df)]
    if not valid:
        return pd.DataFrame()
    merged = valid[0]
    for df in valid[1:]:
        merged = merged.merge(df.drop(columns=["time"], errors="ignore"), on=["video_id", "time_bin"], how="outer")
    merged["time"] = merged["time_bin"] * window_sec
    merged = merged.sort_values(["video_id", "time_bin"]).reset_index(drop=True)
    numeric_cols = [c for c in merged.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    # 広域振動スコアは0.5秒hopなので、0.2秒刻みの行には値がない時刻があります。
    # そのため動画内で前後補完し、次の広域振動スコアが来るまで同じ値を保持します。
    merged[numeric_cols] = merged.groupby("video_id", group_keys=False)[numeric_cols].apply(lambda g: g.ffill().bfill()).fillna(0.0)
    for col in merged.select_dtypes(exclude=[np.number]).columns:
        if col != "video_id":
            merged[col] = merged[col].fillna("unknown")
    return merged


# 中央値と MAD を使い、外れ値に強い正方向 z-score を作ります。
# 関数メモ: robust_positive_zscore
# - 中央値とMADに基づく、正方向だけの頑健 z-score を作ります。
# - 通常より大きい変化だけを振動候補として強調し、小さい側の変動は0に丸めます。

def robust_positive_zscore(values: np.ndarray | pd.Series) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.zeros((0,), dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values, dtype=float)
    median = float(np.median(finite))
    scale = max(1.4826 * float(np.median(np.abs(finite - median))), 1e-6)
    return np.maximum((np.nan_to_num(values, nan=median) - median) / scale, 0.0)


# スコアを指定パーセンタイル範囲で 0〜1 に正規化します。rear だけ極端に跳ねる問題を抑える目的です。
# 関数メモ: percentile_normalize_0_1
# - 指定パーセンタイル範囲を使ってスコアを0〜1へ正規化します。
# - 動画やカメラごとのスケール差を抑え、rear/front の値を比較しやすくします。

def percentile_normalize_0_1(
    values: np.ndarray | pd.Series,
    lower_percentile: float = DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
    upper_percentile: float = DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
    positive_only: bool = True,
) -> np.ndarray:
    values_arr = np.asarray(values, dtype=float)
    out = np.zeros_like(values_arr, dtype=float)
    finite_mask = np.isfinite(values_arr)
    if not finite_mask.any():
        return out
    fit_values = values_arr[finite_mask]
    if positive_only:
        fit_values = fit_values[fit_values > FLOW_SCORE_MIN_VISIBLE]
    if fit_values.size == 0:
        return out
    lo_q, hi_q = sorted([float(np.clip(lower_percentile, 0.0, 100.0)), float(np.clip(upper_percentile, 0.0, 100.0))])
    lo, hi = float(np.percentile(fit_values, lo_q)), float(np.percentile(fit_values, hi_q))
    if hi <= lo:
        out[finite_mask] = np.where(values_arr[finite_mask] > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        out[finite_mask] = np.clip((values_arr[finite_mask] - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return np.where(values_arr > FLOW_SCORE_MIN_VISIBLE, out, 0.0).astype(float) if positive_only else out.astype(float)


# flow ベクトルの向き変化量 |dtheta| を計算します。小さすぎるベクトルは向きが不安定なので0扱いします。
# 関数メモ: compute_flow_direction_change_abs
# - グリッドごとの flow x/y から、連続時刻の向き変化量 |dtheta| を計算します。
# - ベクトルが小さすぎる時刻は向きが不安定なので、振動スコアから除外します。

def compute_flow_direction_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    x = ordered[x_col].astype(float).to_numpy()
    y = ordered[y_col].astype(float).to_numpy()
    magnitude = np.hypot(x, y)
    angle = np.arctan2(y, x)
    delta = np.diff(angle, prepend=angle[:1])
    direction_change = np.abs(np.arctan2(np.sin(delta), np.cos(delta)))
    valid = (magnitude >= FLOW_DIRECTION_MIN_MAG) & (np.r_[magnitude[:1], magnitude[:-1]] >= FLOW_DIRECTION_MIN_MAG)
    return pd.Series(np.where(valid, np.clip(direction_change, 0.0, np.pi), 0.0), index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


# 振動候補の high/low 閾値を、固定値またはパーセンタイルから決定します。
# 関数メモ: resolve_direction_jitter_thresholds
# - 方向変化スコアから high/low 閾値を固定値またはパーセンタイルで決めます。
# - 動画ごとの分布に合わせて、振動イベント候補の感度を調整します。

def resolve_direction_jitter_thresholds(scores: np.ndarray | pd.Series) -> tuple[float, float]:
    scores_arr = np.asarray(scores, dtype=float)
    positive = scores_arr[np.isfinite(scores_arr) & (scores_arr > FLOW_SCORE_MIN_VISIBLE)]
    if DIRECTION_JITTER_THRESHOLD_MODE == "percentile" and positive.size:
        high = float(np.percentile(positive, np.clip(DIRECTION_JITTER_HIGH_PERCENTILE, 0.0, 100.0)))
        low = float(np.percentile(positive, np.clip(DIRECTION_JITTER_LOW_PERCENTILE, 0.0, 100.0)))
    else:
        high = float(DIRECTION_JITTER_HIGH_THRESHOLD)
        low = float(DIRECTION_JITTER_LOW_THRESHOLD)
    return high, min(low, high)


# high 閾値を超えた窓を起点に、隣接する low 閾値以上の窓も同じ振動イベントとして拾います。
# 関数メモ: mark_direction_jitter_hysteresis_windows
# - high 閾値を超えた窓を起点に、隣接する low 閾値以上の窓もイベントとして選びます。
# - ピーク1点だけでなく、その前後のまとまった振動区間を拾うためのヒステリシス処理です。

def mark_direction_jitter_hysteresis_windows(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.sort_values("window_start_sec").copy()
    scores = np.nan_to_num(window_df["direction_jitter_score"].to_numpy(dtype=float), nan=-np.inf)
    high, low = resolve_direction_jitter_thresholds(scores)
    seed_mask = (scores >= high) & (scores > FLOW_SCORE_MIN_VISIBLE)
    low_mask = (scores >= low) & (scores > FLOW_SCORE_MIN_VISIBLE)
    selected = np.zeros(len(window_df), dtype=bool)
    for seed_index in np.flatnonzero(seed_mask):
        selected[seed_index] = True
        left = seed_index - 1
        while left >= 0 and low_mask[left]:
            selected[left] = True
            left -= 1
        right = seed_index + 1
        while right < len(window_df) and low_mask[right]:
            selected[right] = True
            right += 1
    window_df["direction_jitter_seed"] = seed_mask.astype(bool)
    window_df["direction_jitter_selected"] = selected.astype(bool)
    window_df["direction_jitter_high_threshold"] = high
    window_df["direction_jitter_low_threshold"] = low
    return window_df


# 関数メモ: add_direction_jitter_score_alpha
# - 選択された振動窓に、可視化用の透明度 alpha を付与します。
# - スコアが高いほど濃く描けるようにし、候補区間の強弱を図へ反映します。

def add_direction_jitter_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    if window_df.empty or "direction_jitter_score" not in window_df.columns:
        window_df = window_df.copy()
        window_df["direction_jitter_seed"] = False
        window_df["direction_jitter_selected"] = False
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df
    window_df = mark_direction_jitter_hysteresis_windows(window_df)
    selected_scores = window_df.loc[window_df["direction_jitter_selected"], "direction_jitter_score"].to_numpy(dtype=float)
    finite = selected_scores[np.isfinite(selected_scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= FLOW_SCORE_MIN_VISIBLE:
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df
    lo, hi = float(np.nanmin(finite)), float(np.nanmax(finite))
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    normalized = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), 1.0, 0.0) if hi <= lo else np.clip((np.nan_to_num(scores, nan=lo) - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    window_df["direction_jitter_score_alpha"] = np.where(
        window_df["direction_jitter_selected"].to_numpy(dtype=bool),
        FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN),
        0.0,
    ).astype(float)
    return window_df


# 1グリッドについて、1秒窓ごとの向き変化統計と振動スコアを作ります。
# 関数メモ: build_flow_direction_jitter_window_table
# - 1つの camera/grid について、1秒窓ごとの方向変化統計と振動スコアを作ります。
# - 複数指標を合成して direction_jitter_score を作り、グリッド別の局所振動を評価します。

def build_flow_direction_jitter_window_table(raw_flow_df: pd.DataFrame, camera: str, gy: int, gx: int) -> pd.DataFrame:
    x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
    y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
    if raw_flow_df.empty or x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    direction_change = compute_flow_direction_change_abs(ordered, x_col, y_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = max(float(np.nanmax(times)) if np.isfinite(times).any() else 0.0, FLOW_SCORE_WINDOW_SEC)

    rows = []
    # raw flow の最終時刻から、1.0秒窓を0.5秒刻みで置く開始時刻を作ります。
    # 動画が短くても少なくとも 0.0 秒の1窓は作り、後段の列構造を保ちます。
    max_start = max(float(duration_sec) - float(FLOW_SCORE_WINDOW_SEC), 0.0)
    starts = np.arange(0.0, max_start + 1e-9, float(FLOW_SCORE_HOP_SEC), dtype=np.float32).tolist()
    window_starts = [float(value) for value in (starts if starts else [0.0])]
    for start_sec in window_starts:
        end_sec = float(min(start_sec + FLOW_SCORE_WINDOW_SEC, duration_sec))
        start_sec = max(end_sec - FLOW_SCORE_WINDOW_SEC, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            mask = np.zeros_like(times, dtype=bool)
            mask[int(np.argmin(np.abs(times - start_sec)))] = True
        segment = direction_change[mask]
        direction_max = float(np.max(segment))
        high_ratio = float(np.mean(segment >= direction_max * FLOW_SCORE_HIGH_RATIO_FRACTION)) if direction_max > FLOW_SCORE_MIN_VISIBLE else 0.0
        rows.append({
            "camera": camera,
            "grid_x": gx + 1,
            "grid_y": gy + 1,
            "grid_col": gx,
            "grid_row": gy,
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "direction_mean": float(np.mean(segment)),
            "direction_sum": float(np.sum(segment)),
            "direction_p95": float(np.percentile(segment, 95)),
            "direction_max": direction_max,
            "direction_std": float(np.std(segment)),
            "direction_variation": float(np.mean(np.abs(np.diff(segment)))) if segment.size >= 2 else 0.0,
            "direction_high_ratio": high_ratio,
        })

    window_df = pd.DataFrame(rows)
    for name in ["direction_sum", "direction_high_ratio", "direction_variation", "direction_p95"]:
        window_df[f"{name}_z"] = robust_positive_zscore(window_df[name])
        # 複数の向き変化指標を重み付きで足し、ギザギザした揺れを1つの raw スコアにします。
    window_df["direction_jitter_score_raw"] = (
        0.35 * window_df["direction_sum_z"]
        + 0.25 * window_df["direction_high_ratio_z"]
        + 0.25 * window_df["direction_variation_z"]
        + 0.15 * window_df["direction_p95_z"]
    ).astype(float)
        # raw スコアは動画・カメラごとのスケール差が出やすいため、0〜1へ正規化します。
    window_df["direction_jitter_score"] = percentile_normalize_0_1(window_df["direction_jitter_score_raw"])
    return add_direction_jitter_score_alpha(window_df)


# 関数メモ: build_flow_direction_jitter_score_table
# - 指定 camera の全グリッドに対して direction jitter の窓表を作って結合します。
# - 3x3 のどこで振動が出たかを、camera 単位で一覧化します。

def build_flow_direction_jitter_score_table(raw_flow_df: pd.DataFrame, camera: str) -> pd.DataFrame:
    tables = [
        build_flow_direction_jitter_window_table(raw_flow_df, camera, gy, gx)
        for gy in range(FLOW_GRID[0])
        for gx in range(FLOW_GRID[1])
    ]
    tables = [df for df in tables if len(df)]
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


# 同じ時刻のグリッドをスコア順に並べ、上位 K 個を可視化・集計できるよう印を付けます。
# 関数メモ: add_direction_jitter_topk_columns
# - 同じ時刻のグリッドを振動スコア順に並べ、上位K件に rank と選択フラグを付けます。
# - 広域振動スコアを作るときに、画面全体のうち強く揺れた領域だけを平均するために使います。

def add_direction_jitter_topk_columns(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required = {"video_id", "camera", "window_start_sec", "direction_jitter_score"}
    missing = sorted(required - set(direction_jitter_df.columns))
    if missing:
        raise ValueError(f"Missing columns for direction jitter top-k: {missing}")

    work = direction_jitter_df.copy()
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)
    scores = pd.to_numeric(work["direction_jitter_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work["thresholded_direction_jitter_score"] = np.where(selected, scores, 0.0)
    work["direction_jitter_topk_rank"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
    work["direction_jitter_topk_selected"] = False
    work["window_start_bin"] = np.round(work["window_start_sec"].astype(float) / max(FLOW_SCORE_HOP_SEC, 1e-6)).astype(int)

    meta_cols = [c for c in ["target_label", "target_category", "target_environment"] if c in work.columns]
    for _, group in work.groupby(["video_id", *meta_cols, "camera", "window_start_bin"], sort=False):
        ranked = group[group["thresholded_direction_jitter_score"] > FLOW_SCORE_MIN_VISIBLE]
        ranked = ranked.sort_values("thresholded_direction_jitter_score", ascending=False).head(max(1, int(top_k)))
        if len(ranked):
            work.loc[ranked.index, "direction_jitter_topk_rank"] = np.arange(1, len(ranked) + 1, dtype=int)
            work.loc[ranked.index, "direction_jitter_topk_selected"] = True
    return work


# カメラごと・時刻ごとに、閾値通過後の上位 K グリッドを平均して広域振動スコアにします。
# 関数メモ: build_broad_vibration_score_table
# - camera/time ごとに上位Kグリッドの振動スコアを平均し、広域振動スコアを作ります。
# - 局所的な強い揺れを拾いつつ、単一グリッドのノイズに引っ張られすぎない形にします。

def build_broad_vibration_score_table(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    # grid 別の振動スコアを作ったあと、同じ camera + 同じ時刻窓で上位K件を平均します。
    # これにより「画面のどこか一部だけが強く揺れた」ケースも motion 特徴として拾えます。
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    work = add_direction_jitter_topk_columns(direction_jitter_df, top_k=top_k)
    meta_cols = [c for c in ["target_label", "target_category", "target_environment"] if c in work.columns]
    rows = []
    # window_start_bin は 0.5秒hop側の時刻IDです。ここではまだ0.2秒刻みには変換しません。
    for keys, group in work.groupby(["video_id", *meta_cols, "camera", "window_start_bin"], sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        video_id = keys[0]
        meta_values = dict(zip(meta_cols, keys[1:1 + len(meta_cols)]))
        camera = keys[1 + len(meta_cols)]
        window_start_bin = keys[2 + len(meta_cols)]
        values = np.sort(np.nan_to_num(group["thresholded_direction_jitter_score"].to_numpy(dtype=float), nan=0.0))[::-1]
        top_values = np.zeros(max(1, int(top_k)), dtype=float)
        top_values[: min(top_values.size, values.size)] = values[: min(top_values.size, values.size)]
        rows.append({
            "video_id": video_id,
            **meta_values,
            "camera": camera,
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "window_center_sec": float(group["window_center_sec"].median()),
            "broad_vibration_score": float(top_values.mean()),
            "broad_vibration_top_k": int(top_values.size),
            "selected_grid_count": int(group.get("direction_jitter_selected", pd.Series(False, index=group.index)).astype(bool).sum()),
            "nonzero_top_k_count": int(np.count_nonzero(top_values > FLOW_SCORE_MIN_VISIBLE)),
            "max_thresholded_direction_jitter_score": float(np.max(values)) if values.size else 0.0,
            **{f"broad_vibration_top{i}_score": float(v) for i, v in enumerate(top_values, start=1)},
        })
    return pd.DataFrame(rows).sort_values(["video_id", "camera", "window_start_sec"]).reset_index(drop=True)


# 広域振動スコア表を、モデル入力に使いやすい front/rear の2列へ pivot します。
# 関数メモ: build_broad_vibration_feature_df
# - grid別 direction jitter から、モデル入力用の front/rear 広域振動特徴列を作ります。
# - motion モデルへ入る主特徴である `front_broad_vibration_score` / `rear_broad_vibration_score` を生成します。

def build_broad_vibration_feature_df(raw_flow_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    # 広域振動スコアを front/rear の2列に pivot し、モデル入力用の特徴表に変換します。
    empty_cols = ["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS]
    if raw_flow_df is None or raw_flow_df.empty:
        return pd.DataFrame(columns=empty_cols)

    raw_flow_df = raw_flow_df.copy()
    if "video_id" not in raw_flow_df.columns:
        raw_flow_df["video_id"] = "unknown"
    direction_tables = []
    # video_id ごとに処理し、複数動画をまとめて渡しても時系列が混ざらないようにします。
    for video_id, video_df in raw_flow_df.groupby("video_id", sort=False):
        meta = {c: video_df[c].dropna().iloc[0] for c in ["target_label", "target_category", "target_environment"] if c in video_df.columns and video_df[c].notna().any()}
        for camera in ["front", "rear"]:
            # cameraごとに 3x3 グリッドの向き変化スコアを作ります。
            direction_df = build_flow_direction_jitter_score_table(video_df, camera)
            if len(direction_df):
                direction_df.insert(0, "video_id", video_id)
                for col, value in meta.items():
                    direction_df[col] = value
                direction_tables.append(direction_df)
    if not direction_tables:
        return pd.DataFrame(columns=empty_cols)

    broad_df = build_broad_vibration_score_table(pd.concat(direction_tables, ignore_index=True), top_k=top_k)
    if broad_df.empty:
        return pd.DataFrame(columns=empty_cols)

    broad_df = broad_df.copy()
    # 広域振動スコアは0.5秒hopですが、モデル入力は0.2秒刻みです。
    # まず window_start_sec を最も近い 0.2秒 time_bin に割り当て、
    # align_features_by_time 側で前後補完して0.2秒行へなじませます。
    broad_df["time"] = broad_df["window_start_sec"].astype(float)
    broad_df["time_bin"] = np.round(broad_df["time"] / max(WINDOW_SEC, 1e-6)).astype(int)
    feature_df = broad_df.pivot_table(
        index=["video_id", "time_bin", "time"],
        columns="camera",
        values="broad_vibration_score",
        aggfunc="mean",
    ).reset_index().rename(columns={"front": "front_broad_vibration_score", "rear": "rear_broad_vibration_score"})
    feature_df.columns.name = None
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in feature_df.columns:
            feature_df[col] = 0.0
    return feature_df[empty_cols].copy()


# 広域振動スコア列が欠けた場合でも、モデル入力の列が必ず存在するよう補完します。
# 関数メモ: ensure_broad_vibration_columns
# - 広域振動スコア列が欠けている場合に0で補完し、存在する場合は動画内で前後補完します。
# - 入力動画に片側カメラがない・計算に失敗した場合でも、モデル入力列を安定させます。

def ensure_broad_vibration_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
        else:
            values = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[col] = values.groupby(df["video_id"]).transform(lambda s: s.ffill().bfill()).fillna(0.0) if "video_id" in df.columns else values.fillna(0.0)
    return df


MODEL_EXCLUDE_COLS = {
    "time", "time_bin", "video_id", "sample_id", "label",
    "front_path", "rear_path", "audio_path",
}


# 列名から特徴量グループを判定します。ここで audio と broad_vibration だけをモデル対象にします。
# 関数メモ: infer_feature_group
# - 列名から audio_basic / audio_mel / broad_vibration / metadata などの特徴グループを判定します。
# - モデルごとの入力列選択や、特徴量カタログ作成の基準になります。

def infer_feature_group(column: str) -> str:
    if column in MODEL_EXCLUDE_COLS:
        return "metadata"
    if column in BROAD_VIBRATION_COLUMNS or column.endswith("_broad_vibration_score"):
        return "broad_vibration"
    if column.startswith("audio_mel_"):
        return "audio_mel"
    if column.startswith("audio_"):
        return "audio_basic"
    return "other"


# 関数メモ: build_feature_catalog
# - 特徴量列ごとにグループ、使用可否、dtype、非欠損数をまとめたカタログを作ります。
# - モデルが何を見ているかをCSVで確認・監査できるようにします。

def build_feature_catalog(features_df: pd.DataFrame, feature_groups: dict[str, bool] = FEATURE_GROUPS) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "column": col,
            "group": infer_feature_group(col),
            "enabled": bool(feature_groups.get(infer_feature_group(col), False)) and col not in FEATURE_EXCLUDE_COLUMNS,
            "dtype": str(features_df[col].dtype),
            "non_null": int(features_df[col].notna().sum()),
        }
        for col in features_df.columns
    ])


# モデル設定で有効になっているグループの列だけを学習入力として選びます。
# 関数メモ: select_model_source_columns
# - モデル設定で有効になっている特徴グループだけを入力列として選びます。
# - audio モデルと motion モデルで見る列を明確に分けるためのフィルタです。

def select_model_source_columns(features_df: pd.DataFrame, feature_groups: dict[str, bool] = FEATURE_GROUPS) -> list[str]:
    return [
        col for col in features_df.columns
        if infer_feature_group(col) != "metadata" and col not in FEATURE_EXCLUDE_COLUMNS and feature_groups.get(infer_feature_group(col), False)
    ]


# 関数メモ: build_expanded_feature_group_map
# - one-hot 展開後の列名を、元の特徴グループへ対応付けます。
# - 展開後の各列にグループ重みを適用するための辞書を作ります。

def build_expanded_feature_group_map(source_columns: list[str], expanded_columns: list[str]) -> dict[str, str]:
    source_groups = {col: infer_feature_group(col) for col in source_columns}
    group_map: dict[str, str] = {}
    for expanded_col in expanded_columns:
        if expanded_col in source_groups:
            group_map[expanded_col] = source_groups[expanded_col]
            continue
        matched = max((col for col in source_columns if expanded_col.startswith(f"{col}_")), key=len, default=None)
        group_map[expanded_col] = source_groups.get(matched, "other")
    return group_map


# 関数メモ: build_feature_weight_vector
# - 展開後特徴量ごとに、所属グループに応じた重みベクトルを作ります。
# - 入力空間で audio / motion などのグループ寄与を調整できるようにします。

def build_feature_weight_vector(
    expanded_feature_names: np.ndarray,
    expanded_feature_group_map: dict[str, str],
    feature_group_weights: dict[str, float] = FEATURE_GROUP_WEIGHTS,
) -> np.ndarray:
    return np.asarray([
        float(feature_group_weights.get(expanded_feature_group_map.get(str(name), "other"), 1.0))
        for name in expanded_feature_names
    ], dtype=float)


## 10. 1サンプルの特徴量抽出

まず1本で特徴量の形と可視化を確認します。

In [ ]:
# ------------------------------------------------------------
# セル概要: 1サンプル特徴抽出と学習対象選択
# ------------------------------------------------------------
# - 1本のサンプルから全特徴を作る関数と、学習に使う normal サンプルの選択処理を定義します。
# - manual / random のどちらでも、後で推論側が除外できるよう train_sample_list.csv を保存します。
# ------------------------------------------------------------

# ============================================================
# 1サンプルの特徴量作成と学習サンプル選択
# ------------------------------------------------------------
# 1つの動画ペアから音声特徴、front/rear raw flow、広域振動スコアを作り、
# time_bin 単位で結合します。後半では学習に使う正常動画リストを管理します。
# ============================================================
# 関数メモ: extract_sample_features
# - 1サンプルの audio/front/rear を読み、音声特徴・raw flow・広域振動特徴を結合して返します。
# - 学習ノートでも推論ノートでも、1本分のモデル入力表を作る中心関数です。

def extract_sample_features(sample: pd.Series) -> pd.DataFrame:
    video_id = str(sample["sample_id"])
    # 後で time_bin で結合するため、各特徴 DataFrame をここへ集めます。
    feature_dfs = []
    # 広域振動スコアは raw flow の 3x3 グリッド x/y から作るため、別途保持します。
    raw_flow_dfs = []

    if sample.get("audio_path") is not pd.NA and Path(sample["audio_path"]).exists():
        feature_dfs.append(extract_audio_features(sample["audio_path"], video_id=video_id))

    # front/rear を同じ処理で読み、カメラごとの flow 特徴を作ります。
    for prefix, enabled, path_col in [("front", USE_FRONT, "front_path"), ("rear", USE_REAR, "rear_path")]:
        if not enabled or sample.get(path_col) is pd.NA or not Path(sample[path_col]).exists():
            continue
        frames = extract_video_frames(sample[path_col], fps_sample=FPS_SAMPLE, resize_width=FRAME_RESIZE_WIDTH)
        flow_df = compute_optical_flow_features(frames, prefix, video_id=video_id)
        feature_dfs.append(flow_df)
        raw_flow_dfs.append(flow_df)

    if raw_flow_dfs:
        raw_flow_df = raw_flow_dfs[0]
        for other in raw_flow_dfs[1:]:
            raw_flow_df = raw_flow_df.merge(other.drop(columns=["time_bin"], errors="ignore"), on=["video_id", "time"], how="outer")
        raw_flow_df = raw_flow_df.sort_values(["video_id", "time"]).reset_index(drop=True)
        # front/rear の raw flow を横結合してから、広域振動スコア2列を作ります。
        feature_dfs.append(build_broad_vibration_feature_df(raw_flow_df))

    # audio は0.2秒窓、広域振動スコアは0.5秒hopなので、
    # ここで video_id + time_bin にそろえます。広域振動スコアがない行は
    # 前後補完され、0.2秒刻みのモデル入力として扱える形になります。
    df = ensure_broad_vibration_columns(align_features_by_time(feature_dfs))
    df["environment"] = sample.get("environment", "unknown")
    df["label"] = "normal"
    return df


# 学習対象は normal のみです。ここから下は、normal の中でどの動画を学習に使うかを決めます。
# 関数メモ: normalize_ratio_dict
# - 指定された比率辞書を、対象ラベルだけの合計1.0の比率へ正規化します。
# - 未指定ラベルや合計0の設定でも、ランダム抽出が破綻しないようにします。

def normalize_ratio_dict(ratio: dict[str, float], labels: list[str]) -> dict[str, float]:
    # 比率は合計1でなくてもよいので、正の値だけを取り出して正規化します。
    values = {label: max(float(ratio.get(label, 0.0)), 0.0) for label in labels}
    total = sum(values.values())
    if total <= 0:
        return {label: 1.0 / max(len(labels), 1) for label in labels}
    return {label: value / total for label, value in values.items()}


# 関数メモ: allocate_counts_by_ratio
# - 目標件数を比率に従って整数件数へ配分します。
# - 候補数が不足するラベルは上限で止め、余った枠を他ラベルへ再配分します。

def allocate_counts_by_ratio(available_counts: dict[str, int], ratio: dict[str, float], target_count: int) -> dict[str, int]:
    # 指定比率を整数件数に変換します。候補数が足りない環境は候補数で頭打ちにし、
    # 余った枠はまだ候補が残っている環境へ配ります。
    available_counts = {str(label): int(count) for label, count in available_counts.items() if int(count) > 0}
    if not available_counts:
        return {}
    target_count = min(int(target_count), sum(available_counts.values()))
    ratios = normalize_ratio_dict(ratio, sorted(available_counts))
    raw_targets = {label: target_count * ratios.get(label, 0.0) for label in available_counts}
    quotas = {label: min(int(math.floor(raw_targets[label])), available_counts[label]) for label in available_counts}

    remaining = target_count - sum(quotas.values())
    while remaining > 0:
        expandable = [label for label in available_counts if quotas[label] < available_counts[label]]
        if not expandable:
            break
        expandable.sort(
            key=lambda label: (raw_targets[label] - quotas[label], ratios.get(label, 0.0), available_counts[label]),
            reverse=True,
        )
        quotas[expandable[0]] += 1
        remaining -= 1
    return {label: count for label, count in quotas.items() if count > 0}


# 関数メモ: build_train_sample
# - manual 学習指定の sample_id を usable normal サンプルから1件へ解決します。
# - 同じ sample_id が複数環境にある場合は、environment 指定を促すため例外にします。

def build_train_sample(spec: dict, usable_samples: pd.DataFrame) -> pd.Series:
    # manual モードでは sample_id だけを指定します。category は normal 固定です。
    sample_id = str(spec["sample_id"]).strip()
    matches = usable_samples[usable_samples["sample_id"].astype(str).eq(sample_id)].copy()
    if "environment" in spec:
        matches = matches[matches["environment"].astype(str).eq(str(spec["environment"]))]
    if matches.empty:
        raise FileNotFoundError(f"normal sample_id was not found in usable training samples: {sample_id}")
    if len(matches) > 1:
        choices = matches[["sample_id", "environment"]].astype(str).agg("/".join, axis=1).tolist()
        raise ValueError("sample_id matched multiple normal rows. Add environment to TRAIN_SPECS:\n" + "\n".join(choices))
    return matches.iloc[0]


# 関数メモ: select_random_train_samples
# - usable normal サンプルから、件数と indoor/outdoor 比率に従ってランダムに学習対象を選びます。
# - random_state と環境名由来の seed を使い、同じ設定なら再現可能な選択にします。

def select_random_train_samples(
    usable_samples: pd.DataFrame,
    target_count: int | None = RANDOM_TRAIN_COUNT,
    environment_ratios: dict[str, float] = RANDOM_TRAIN_ENVIRONMENT_RATIOS,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    # normal の usable データから、件数と indoor/outdoor 比率に従ってランダム抽出します。
    if usable_samples.empty:
        return usable_samples.copy()
    if target_count is None or int(target_count) < 0:
        return usable_samples.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    target_total = min(int(target_count), len(usable_samples))
    env_counts = usable_samples["environment"].value_counts().sort_index().to_dict()
    quotas = allocate_counts_by_ratio(env_counts, environment_ratios, target_total)

    selected_parts = []
    for index, (environment, count) in enumerate(sorted(quotas.items())):
        env_df = usable_samples[usable_samples["environment"].astype(str).eq(str(environment))]
        stable_env_seed = sum((i + 1) * ord(ch) for i, ch in enumerate(str(environment)))
        selected_parts.append(env_df.sample(n=count, random_state=random_state + stable_env_seed + index))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else usable_samples.head(0).copy()
    return selected.sample(frac=1.0, random_state=random_state).reset_index(drop=True)


# 関数メモ: save_train_sample_list
# - 今回学習に使った normal サンプル一覧をCSVへ保存します。
# - 推論ノートで random_untrained を選ぶ際、学習済みサンプルを除外するために使います。

def save_train_sample_list(train_samples: pd.DataFrame, sample_list_path: Path) -> None:
    # 推論ノート側で「学習に使っていないデータ」を除外できるよう、選択結果をCSVへ残します。
    sample_list_path.parent.mkdir(parents=True, exist_ok=True)
    cols = [
        "sample_id", "category", "environment", "front_path", "rear_path", "audio_path",
        *[c for c in ["input_duration_sec", "status", "audio_status"] if c in train_samples.columns],
    ]
    out = train_samples[[c for c in cols if c in train_samples.columns]].copy()
    for col in ["front_path", "rear_path", "audio_path"]:
        if col in out.columns:
            out[col] = out[col].astype(str)
    out.to_csv(sample_list_path, index=False)


usable_samples = samples_df[samples_df["usable"]].copy()
usable_samples["sample_id"] = usable_samples["sample_id"].map(lambda value: str(value).strip())
usable_samples["category"] = "normal"

if TRAIN_SELECTION_MODE == "manual":
    if not TRAIN_SPECS:
        raise ValueError("TRAIN_SPECS must contain at least one sample when TRAIN_SELECTION_MODE='manual'.")
    train_samples = pd.DataFrame([build_train_sample(spec, usable_samples) for spec in TRAIN_SPECS]).reset_index(drop=True)
    selection_source = "manual"
elif TRAIN_SELECTION_MODE == "random":
    train_samples = select_random_train_samples(usable_samples, RANDOM_TRAIN_COUNT, RANDOM_TRAIN_ENVIRONMENT_RATIOS, RANDOM_STATE)
    selection_source = "random"
else:
    raise ValueError("TRAIN_SELECTION_MODE must be 'manual' or 'random'.")

save_train_sample_list(train_samples, TRAIN_SAMPLE_LIST_PATH)

print(f"usable normal samples: {len(usable_samples)} / selected for this run: {len(train_samples)}")
print(f"selection mode: {TRAIN_SELECTION_MODE}")
print(f"selection source: {selection_source}")
print(f"train sample list: {TRAIN_SAMPLE_LIST_PATH}")
display(train_samples["environment"].value_counts().rename_axis("environment").reset_index(name="selected_count"))
display(train_samples[["sample_id", "category", "environment", "front_path", "rear_path", "audio_path"]].head(20))


In [ ]:
# ------------------------------------------------------------
# セル概要: 先頭サンプルでの動作確認
# ------------------------------------------------------------
# - 本格学習の前に1本だけ特徴量抽出し、行数・列数・代表統計を確認します。
# - ここで列欠けや値のスケール異常を見つけると、全件処理の手戻りを減らせます。
# ------------------------------------------------------------

# 動作確認用に先頭1本だけ特徴量抽出します。
one_sample = train_samples.iloc[0]
one_features_df = extract_sample_features(one_sample)
print(one_features_df.shape)
display(one_features_df.head())
display(one_features_df.describe().T.head(30))

## 11. 特徴量の簡易可視化

## 12. 正常データ群から特徴量抽出

件数を増やすほど時間がかかります。最初は `MAX_TRAIN_VIDEOS=8` 程度で確認してください。

In [ ]:
# ------------------------------------------------------------
# セル概要: 正常データ群の特徴抽出
# ------------------------------------------------------------
# - 選択済み normal サンプルすべてに特徴量抽出を適用し、学習用 DataFrame にまとめます。
# - 失敗した動画があっても全体処理が止まらないよう、動画単位で例外を警告にしています。
# ------------------------------------------------------------

# ============================================================
# 正常データ群から特徴量抽出
# ------------------------------------------------------------
# 選択済みの正常動画すべてに対して特徴量を作り、学習用 DataFrame にまとめます。
# ============================================================
all_feature_dfs = []
# 1本でも失敗して全体が止まらないよう、動画単位で try/except しています。
for _, sample in tqdm(train_samples.iterrows(), total=len(train_samples), desc="extract features"):
    try:
        all_feature_dfs.append(extract_sample_features(sample))
    except Exception as exc:
        warnings.warn(f"feature extraction failed for sample_id={sample.get('sample_id')}: {exc}")

# 全動画の特徴量を縦に結合します。1行は 0.2 秒窓の1サンプルです。
# この段階の features_df が、そのまま audio/motion モデルの学習母集団になります。
features_df = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()
features_df = ensure_broad_vibration_columns(features_df)

print(features_df.shape)
print("broad vibration columns used by motion model:", [c for c in BROAD_VIBRATION_COLUMNS if c in features_df.columns])
if len(features_df):
    display(features_df[["video_id", "time", *BROAD_VIBRATION_COLUMNS]].head())
    display(features_df[BROAD_VIBRATION_COLUMNS].describe().T)
display(features_df.head())
# 学習に使った特徴量をCSVにも残します。
# 後から「モデルにどの値が入っていたか」を確認するための監査用ファイルです。
features_df.to_csv(OUTPUT_DIR / "features.csv", index=False)
print(f"saved: {OUTPUT_DIR / 'features.csv'}")


## 13. 前処理と Isolation Forest 学習

In [ ]:
# ------------------------------------------------------------
# セル概要: IsolationForest 学習とスコア合成
# ------------------------------------------------------------
# - audio / motion の2モデルを個別に学習し、それぞれの異常度を0〜1へキャリブレーションします。
# - 音声と動きのピークが近いかを見る sync_score を加え、最終 anomaly_score を作ります。
# ------------------------------------------------------------

# ============================================================
# モデル学習とスコア合成
# ------------------------------------------------------------
# audio / motion それぞれに IsolationForest を学習し、
# 0〜1 にキャリブレーションした異常度と同期スコアを合成します。
# ============================================================
# 関数メモ: preprocess_features
# - モデル入力列を選択し、one-hot化・欠損補完・標準化・グループ重み適用まで行います。
# - 推論時に同じ変換を再利用するため、pipeline と列順を artifact に保存します。

def preprocess_features(
    features_df: pd.DataFrame,
    feature_groups: dict[str, bool] = FEATURE_GROUPS,
    feature_group_weights: dict[str, float] = FEATURE_GROUP_WEIGHTS,
):
    # どの列がどの特徴量グループに属するかを先に整理します。
    # artifact に保存しておくことで、推論側でも「このモデルは何を見ているか」を確認できます。
    feature_catalog = build_feature_catalog(features_df, feature_groups=feature_groups)
    # SCORE_MODEL_CONFIGS に従い、このモデルへ入れる列だけを選びます。
    selected_source_cols = select_model_source_columns(features_df, feature_groups=feature_groups)
    if not selected_source_cols:
        raise ValueError("No feature columns were selected for this model. Check SCORE_MODEL_CONFIGS.")

    # モデルへ渡す表は selected_source_cols だけに絞ります。
    # 現在の motion モデルでは front/rear の広域振動スコア2列だけになります。
    model_df = features_df[selected_source_cols].copy()
    # カテゴリ列が混ざっても扱えるよう one-hot 化します。現在は主に数値列です。
    model_df = pd.get_dummies(model_df, columns=model_df.select_dtypes(include=["object", "category"]).columns, dummy_na=True)
    model_df = model_df.replace([np.inf, -np.inf], np.nan)

    # 欠損補完と標準化は学習 artifact に保存し、推論時にも同じ変換を使います。
    # 学習時と推論時でスケールがずれると異常スコアが変わるため、ここは必ず保存対象です。
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_scaled = pipeline.fit_transform(model_df)
    feature_names = model_df.columns.to_numpy()
    expanded_feature_group_map = build_expanded_feature_group_map(selected_source_cols, feature_names.tolist())
    feature_weight_vector = build_feature_weight_vector(feature_names, expanded_feature_group_map, feature_group_weights)
    X = X_scaled * feature_weight_vector
    return X, X_scaled, pipeline, feature_names, model_df, feature_catalog, selected_source_cols, expanded_feature_group_map, feature_weight_vector


# 関数メモ: train_isolation_forest
# - 正常データだけで IsolationForest を学習します。
# - decision_function の符号を後で反転し、「大きいほど異常」のスコアとして扱います。

def train_isolation_forest(X: np.ndarray, contamination=CONTAMINATION, random_state: int = RANDOM_STATE) -> IsolationForest:
    # 正常データだけを学習し、「正常分布から外れる窓」を高スコアにします。
    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        max_samples="auto",
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X)
    return model


# IsolationForest の raw score を比較しやすい 0〜1 スコアへ写像する分位点を保存します。
# 関数メモ: fit_score_calibration
# - raw anomaly score を0〜1へ写像するための下限・上限分位点を学習データから決めます。
# - モデルや特徴量ごとの raw score 尺度差を抑え、audio/motion を合成しやすくします。

def fit_score_calibration(scores: np.ndarray, quantiles: tuple[float, float] = SCORE_CALIBRATION_QUANTILES) -> dict[str, float]:
    finite = np.asarray(scores, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return {"lower": 0.0, "upper": 1.0}
    lower_q, upper_q = quantiles
    lower = float(np.quantile(finite, lower_q))
    upper = float(np.quantile(finite, upper_q))
    if not np.isfinite(upper) or upper <= lower:
        upper = lower + 1e-6
    return {"lower": lower, "upper": upper}


# 関数メモ: apply_score_calibration
# - 保存済み分位点を使って raw score を0〜1へクリップ正規化します。
# - 学習時・推論時で同じキャリブレーションを使うことでスコアの意味をそろえます。

def apply_score_calibration(scores: np.ndarray | pd.Series, calibration: dict[str, float]) -> np.ndarray:
    values = np.asarray(scores, dtype=float)
    lower = float(calibration.get("lower", 0.0))
    upper = float(calibration.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((values - lower) / denom, 0.0, 1.0)


# 関数メモ: compute_temporal_sync_score
# - 音声異常度と動き異常度が近い時刻に同時に高いかを評価します。
# - 衝撃音だけ・動きだけより、両方が短時間内に出た窓を高く見積もります。

def compute_temporal_sync_score(
    scored_df: pd.DataFrame,
    audio_col: str = SYNC_SCORE_CONFIG["audio_score_column"],
    motion_col: str = SYNC_SCORE_CONFIG["motion_score_column"],
    max_lag_windows: int = SYNC_SCORE_CONFIG["max_lag_windows"],
) -> pd.Series:
    if audio_col not in scored_df.columns or motion_col not in scored_df.columns:
        return pd.Series(0.0, index=scored_df.index, name="sync_score")

    sync = pd.Series(0.0, index=scored_df.index, name="sync_score")
    sort_col = "time_bin" if "time_bin" in scored_df.columns else "time"
    window = max(1, int(max_lag_windows) * 2 + 1)

    # 動画ごとに、音声ピークと動きピークが近い時間にあるかを rolling max で見ます。
    for _, group in scored_df.sort_values(["video_id", sort_col]).groupby("video_id", sort=False):
        audio = group[audio_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        motion = group[motion_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        nearby_audio = audio.rolling(window=window, center=True, min_periods=1).max()
        nearby_motion = motion.rolling(window=window, center=True, min_periods=1).max()
        aligned = np.maximum(audio * nearby_motion, motion * nearby_audio)
        sync.loc[group.index] = np.sqrt(np.clip(aligned, 0.0, 1.0))
    return sync


# 関数メモ: add_composite_scores
# - audio / motion / sync の各スコアを重み付きで合成し、最終 anomaly_score を作ります。
# - 互換用に anomaly_score_smooth も追加し、グラフで見やすい移動平均を持たせます。

def add_composite_scores(
    scored_df: pd.DataFrame,
    final_score_weights: dict[str, float] = FINAL_SCORE_WEIGHTS,
    sync_config: dict = SYNC_SCORE_CONFIG,
) -> pd.DataFrame:
    scored_df = scored_df.copy()
    scored_df["sync_score"] = compute_temporal_sync_score(
        scored_df,
        audio_col=sync_config.get("audio_score_column", "audio_anomaly_score"),
        motion_col=sync_config.get("motion_score_column", "motion_anomaly_score"),
        max_lag_windows=int(sync_config.get("max_lag_windows", 2)),
    )

    # 存在するスコア列だけを使って、重み付き平均で最終スコアを作ります。
    available_weights = {col: float(weight) for col, weight in final_score_weights.items() if col in scored_df.columns and float(weight) > 0}
    weight_sum = sum(available_weights.values())
    if weight_sum <= 0:
        scored_df["final_anomaly_score"] = 0.0
    else:
        final_score = np.zeros(len(scored_df), dtype=float)
        for col, weight in available_weights.items():
            final_score += scored_df[col].astype(float).fillna(0.0).to_numpy() * (weight / weight_sum)
        scored_df["final_anomaly_score"] = np.clip(final_score, 0.0, 1.0)

    # 既存出力との互換用。以後の anomaly_score は合成後の最終スコアです。
    scored_df["anomaly_score"] = scored_df["final_anomaly_score"]
    scored_df["anomaly_score_smooth"] = scored_df.groupby("video_id")["anomaly_score"].transform(
        lambda s: s.rolling(window=5, center=True, min_periods=1).mean()
    )
    return scored_df


# 関数メモ: train_score_model
# - 1つのスコアモデル設定に対して前処理、IsolationForest 学習、スコアキャリブレーションを実行します。
# - モデル本体だけでなく、推論に必要な前処理情報を artifact としてまとめます。

def train_score_model(model_name: str, config: dict, features_df: pd.DataFrame) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray]:
    feature_groups = config.get("feature_groups", FEATURE_GROUPS)
    feature_group_weights = config.get("feature_group_weights", FEATURE_GROUP_WEIGHTS)
    (
        X,
        X_scaled_unweighted,
        preprocess_pipeline,
        feature_names,
        model_input_df,
        feature_catalog,
        selected_source_cols,
        expanded_feature_group_map,
        feature_weight_vector,
    ) = preprocess_features(features_df, feature_groups=feature_groups, feature_group_weights=feature_group_weights)
    model = train_isolation_forest(X)
    # IsolationForest は decision_function が「大きいほど正常」なので、
    # 符号を反転して「大きいほど異常」の raw score として扱います。
    raw_scores = -model.decision_function(X)
    score_calibration = fit_score_calibration(raw_scores)
    calibrated_scores = apply_score_calibration(raw_scores, score_calibration)
    # 推論ノートでは、この artifact 内の pipeline / feature_names / calibration を使って
    # 学習時と同じ前処理・同じ列順でスコアを再現します。
    artifact = {
        "model_name": model_name,
        "model": model,
        "preprocess_pipeline": preprocess_pipeline,
        "feature_names": feature_names,
        "feature_groups": feature_groups,
        "feature_group_weights": feature_group_weights,
        "feature_exclude_columns": FEATURE_EXCLUDE_COLUMNS,
        "selected_source_columns": selected_source_cols,
        "expanded_feature_group_map": expanded_feature_group_map,
        "feature_weight_vector": feature_weight_vector,
        "feature_catalog": feature_catalog,
        "score_calibration": score_calibration,
        "score_column": config.get("score_column", f"{model_name}_anomaly_score"),
        "raw_score_column": config.get("raw_score_column", f"{model_name}_anomaly_score_raw"),
        "model_input_shape": X.shape,
    }
    return artifact, X, raw_scores, calibrated_scores


score_model_artifacts: dict[str, dict] = {}
model_inputs_by_name: dict[str, np.ndarray] = {}
model_feature_catalogs = []

# 設定された score model を順に学習し、各モデルのスコア列を features_df に追加します。
for model_name, config in SCORE_MODEL_CONFIGS.items():
    if not config.get("enabled", True):
        continue
    artifact, X_model, raw_scores, calibrated_scores = train_score_model(model_name, config, features_df)
    score_model_artifacts[model_name] = artifact
    model_inputs_by_name[model_name] = X_model
    features_df[artifact["raw_score_column"]] = raw_scores
    features_df[artifact["score_column"]] = calibrated_scores

    catalog = artifact["feature_catalog"].copy()
    catalog.insert(0, "model_name", model_name)
    model_feature_catalogs.append(catalog)

# audio/motion の個別スコアがそろった後、同期スコアと最終異常スコアを追加します。
features_df = add_composite_scores(features_df)
# feature_catalog は保存用です。どのモデルがどの特徴量グループを使ったかを一覧化します。
feature_catalog_df = pd.concat(model_feature_catalogs, ignore_index=True) if model_feature_catalogs else pd.DataFrame()
primary_model_name = next(iter(score_model_artifacts))
primary_artifact = score_model_artifacts[primary_model_name]
X = model_inputs_by_name[primary_model_name]
feature_names = np.asarray(primary_artifact["feature_names"])

print("score models:", list(score_model_artifacts.keys()))
for name, artifact in score_model_artifacts.items():
    print(f"- {name}: input_shape={artifact['model_input_shape']}, score_calibration={artifact['score_calibration']}")
print("sync score config:", SYNC_SCORE_CONFIG)
print("final score weights:", FINAL_SCORE_WEIGHTS)
summary_cols = ["model_name", "group", "enabled"] if len(feature_catalog_df) else []
if summary_cols:
    display(feature_catalog_df.groupby(summary_cols).size().reset_index(name="column_count"))
    expanded_summaries = []
    for name, artifact in score_model_artifacts.items():
        expanded_summaries.append(pd.DataFrame({
            "model_name": name,
            "feature": artifact["feature_names"],
            "group": [artifact["expanded_feature_group_map"].get(str(c), "other") for c in artifact["feature_names"]],
            "weight": artifact["feature_weight_vector"],
        }))
    display(pd.concat(expanded_summaries, ignore_index=True).groupby(["model_name", "group", "weight"]).size().reset_index(name="expanded_column_count"))
display(features_df[["video_id", "time", "audio_anomaly_score", "motion_anomaly_score", "sync_score", "anomaly_score", "anomaly_score_smooth"]].head())

## 14. 動画単位スコアと上位異常候補

In [ ]:
# ------------------------------------------------------------
# セル概要: 動画単位集計
# ------------------------------------------------------------
# - 0.2秒窓ごとの異常スコアを動画単位に集約し、max / top-k平均 / p95 を確認します。
# - 窓スコアだけでは比較しにくいので、動画全体の異常候補ランキングとして使います。
# ------------------------------------------------------------

# ============================================================
# 動画単位の集計
# ------------------------------------------------------------
# 窓ごとの異常スコアから、動画単位の max / top-k mean / p95 を作ります。
# ============================================================
# 関数メモ: summarize_video_scores
# - 窓ごとのスコアを動画単位の max / top-k平均 / p95 へ集約します。
# - 長い動画同士を比較するため、ピークや上位窓の代表値をランキング用に作ります。

def summarize_video_scores(scored_df: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    def topk_mean(s):
        return s.nlargest(min(top_k, len(s))).mean()

    agg_spec = {
        "max_anomaly_score": ("anomaly_score", "max"),
        "topk_mean_anomaly_score": ("anomaly_score", topk_mean),
        "p95_anomaly_score": ("anomaly_score", lambda s: s.quantile(0.95)),
        "n_windows": ("anomaly_score", "size"),
    }
    for col in ["audio_anomaly_score", "motion_anomaly_score", "sync_score"]:
        if col in scored_df.columns:
            agg_spec[f"max_{col}"] = (col, "max")
            agg_spec[f"topk_mean_{col}"] = (col, topk_mean)
    return scored_df.groupby("video_id").agg(**agg_spec).sort_values("max_anomaly_score", ascending=False).reset_index()

video_scores_df = summarize_video_scores(features_df)
display(video_scores_df)



## 18. 結果保存

In [ ]:
# ------------------------------------------------------------
# セル概要: 成果物保存
# ------------------------------------------------------------
# - スコアCSV、特徴量カタログ、学習済みモデル artifact を保存します。
# - artifact にはモデルだけでなく、前処理 pipeline・列順・重み・settings も含めます。
# ------------------------------------------------------------

# ============================================================
# 学習結果の保存
# ------------------------------------------------------------
# CSV は確認用、joblib は推論ノートで読み込む本体です。
# モデルだけでなく、前処理 pipeline と特徴量設定も一緒に保存します。
# ============================================================
# 関数メモ: save_outputs
# - 学習済みスコア、特徴カタログ、モデル artifact を output_dir へ保存します。
# - 推論ノートが同じ設定・列順・前処理で再現できるよう、必要情報を joblib にまとめます。

def save_outputs(scored_df: pd.DataFrame, output_dir: str | Path = OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    scored_df.to_csv(output_dir / "anomaly_scores.csv", index=False)
    video_scores_df.to_csv(output_dir / "video_scores.csv", index=False)
    feature_catalog_df.to_csv(output_dir / "feature_catalog.csv", index=False)
    for model_name, artifact in score_model_artifacts.items():
        artifact["feature_catalog"].to_csv(output_dir / f"feature_catalog_{model_name}.csv", index=False)

    primary_artifact = score_model_artifacts[next(iter(score_model_artifacts))]
    # 推論時に完全に同じ特徴量定義・前処理でスコア計算するため、必要な設定をまとめて保存します。
    joblib.dump({
        "model_version": "audio_motion_sync_v10_broad_vibration_clean",
        "score_models": score_model_artifacts,
        "score_model_configs": SCORE_MODEL_CONFIGS,
        "sync_score_config": SYNC_SCORE_CONFIG,
        "final_score_weights": FINAL_SCORE_WEIGHTS,
        "plot_score_columns": PLOT_SCORE_COLUMNS,
        "plot_feature_columns": PLOT_FEATURE_COLUMNS,
        "feature_groups": FEATURE_GROUPS,
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
        "feature_exclude_columns": FEATURE_EXCLUDE_COLUMNS,
        "feature_catalog": feature_catalog_df,
        "broad_vibration_columns": BROAD_VIBRATION_COLUMNS,
        "broad_vibration_top_k": BROAD_VIBRATION_TOP_K,
        "settings": {
            "fps_sample": FPS_SAMPLE,
            "window_sec": WINDOW_SEC,
            "audio_sr": AUDIO_SR,
            "n_mels": N_MELS,
            "frame_resize_width": FRAME_RESIZE_WIDTH,
            "flow_analysis_scale": FLOW_ANALYSIS_SCALE,
            "flow_grid": FLOW_GRID,
            "motion_feature": "broad_vibration",
            "flow_score_window_sec": FLOW_SCORE_WINDOW_SEC,
            "flow_score_hop_sec": FLOW_SCORE_HOP_SEC,
            "flow_score_alpha_min": FLOW_SCORE_ALPHA_MIN,
            "flow_score_alpha_max": FLOW_SCORE_ALPHA_MAX,
            "flow_score_min_visible": FLOW_SCORE_MIN_VISIBLE,
            "flow_score_high_ratio_fraction": FLOW_SCORE_HIGH_RATIO_FRACTION,
            "flow_direction_min_mag": FLOW_DIRECTION_MIN_MAG,
            "direction_jitter_high_threshold": DIRECTION_JITTER_HIGH_THRESHOLD,
            "direction_jitter_low_threshold": DIRECTION_JITTER_LOW_THRESHOLD,
            "direction_jitter_threshold_mode": DIRECTION_JITTER_THRESHOLD_MODE,
            "direction_jitter_high_percentile": DIRECTION_JITTER_HIGH_PERCENTILE,
            "direction_jitter_low_percentile": DIRECTION_JITTER_LOW_PERCENTILE,
            "direction_jitter_low_expansion_steps": DIRECTION_JITTER_LOW_EXPANSION_STEPS,
            "direction_jitter_score_lower_percentile": DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
            "direction_jitter_score_upper_percentile": DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
            "broad_vibration_top_k": BROAD_VIBRATION_TOP_K,
            "broad_vibration_columns": BROAD_VIBRATION_COLUMNS,
        },
        "model": primary_artifact["model"],
        "preprocess_pipeline": primary_artifact["preprocess_pipeline"],
        "feature_names": primary_artifact["feature_names"],
        "selected_source_columns": primary_artifact["selected_source_columns"],
        "expanded_feature_group_map": primary_artifact["expanded_feature_group_map"],
        "feature_weight_vector": primary_artifact["feature_weight_vector"],
    }, output_dir / "isolation_forest_feature_poc.joblib")
    print(f"saved outputs under: {output_dir}")


save_outputs(features_df, OUTPUT_DIR)


## 17. 次の検証観点

- 音が大きいところで `audio_rms` と `audio_high_freq_energy` が上がるか
- 衝撃時刻付近で `front_broad_vibration_score` / `rear_broad_vibration_score` が上がるか
- `audio_anomaly_score` と `motion_anomaly_score` のピークが近い時刻で `sync_score` が上がるか
- 正常データだけで学習したとき、通常走行の一部が過剰に高スコアになっていないか
